# Appendix A9-A11: cross-model attribution-score correlation

Reproduces Figures A9 (Automotive), A10 (Career), A11 (Educational) of
`Unequal_influence.pdf` - Spearman correlation between every pair of
models' attribution scores on the same dataset. This is the cheapest item
in the appendix: every model's `attributions.csv` is already sitting in
`cross_model_figure5_<dataset>.yaml`'s results_root once that run
completes, and correlating them needs no retraining, no GPU, not even a new
evaluation - just reading CSVs that already exist.

**Prerequisite**:

```bash
cd finetuning
em-influence run experiments/cross_model_figure5_auto.yaml --resume
em-influence run experiments/cross_model_figure5_career.yaml --resume
em-influence run experiments/cross_model_figure5_edu.yaml --resume
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from em_influence.artifacts import iter_stage_artifacts

RESULTS_ROOTS = {
    "auto": Path("../../results/em_influence/filter_sweep_auto"),
    "career": Path("../../results/em_influence/filter_sweep_career"),
    "edu": Path("../../results/em_influence/filter_sweep_edu"),
}
OUTPUT_DIR = RESULTS_ROOTS["career"] / "plots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_LABELS = {"auto": "Automotive", "career": "Career", "edu": "Educational"}

MODEL_LABELS = {
    "allenai/Olmo-3-7B-Instruct-SFT": "OLMo 3 7B",
    "Qwen/Qwen2.5-1.5B-Instruct": "Qwen 2.5 1B",
    "Qwen/Qwen2.5-3B-Instruct": "Qwen 2.5 3B",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen 2.5 7B",
    "Qwen/Qwen2.5-14B-Instruct": "Qwen 2.5 14B",
    "Qwen/Qwen3-4B": "Qwen 3 4B",
    "Qwen/Qwen3-8B": "Qwen 3 8B",
    "Qwen/Qwen3-14B": "Qwen 3 14B",
    "meta-llama/Llama-3.2-1B-Instruct": "Llama 3.2 1B",
    "meta-llama/Llama-3.2-3B-Instruct": "Llama 3.2 3B",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama 3.1 8B",
}
# Matches cross_model_figure5_<dataset>.yaml's `cross_model.models` order,
# which is itself the paper's Figure 5/A8/A9-A11 model ordering.
MODEL_ORDER = list(MODEL_LABELS)


## Load every model's attribution scores

`iter_stage_artifacts` scans `results_root/artifacts/*/.em_influence.json`
directly - no manifest.csv involved, since that file only tracks `evaluate`
jobs. `cross_model_sweep`'s attribution jobs carry both `model` and `method`
parameters; every model in `cross_model_figure5_<dataset>.yaml` computes
`cosine_similarity` against its own baseline, so filtering on that method
gives exactly one row per model.

In [ ]:
def load_model_attributions(results_root, method="cosine_similarity"):
    scores = {}
    for metadata, artifact_dir in iter_stage_artifacts(results_root, "attribute"):
        params = metadata.parameters or {}
        if params.get("method") != method or "model" not in params:
            continue
        csv_path = artifact_dir / "attributions.csv"
        if not csv_path.is_file():
            continue
        scores[params["model"]] = pd.read_csv(csv_path).sort_values("index_example_idx")["attribution"].to_numpy()
    return scores


## Figures A9 / A10 / A11

In [ ]:
def plot_attribution_correlation(dataset, results_root, model_order=MODEL_ORDER):
    scores = load_model_attributions(results_root)
    models = [model for model in model_order if model in scores]
    corr = pd.DataFrame({model: scores[model] for model in models}).corr(method="spearman").to_numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    labels = [MODEL_LABELS.get(model, model) for model in models]
    ax.set_xticks(range(len(models))); ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_yticks(range(len(models))); ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(models)):
        for j in range(len(models)):
            ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, label="Spearman correlation")
    ax.set_title(f"{DATASET_LABELS.get(dataset, dataset)} attribution-score correlations")
    fig.tight_layout()
    return fig


for dataset in ("auto", "career", "edu"):
    fig = plot_attribution_correlation(dataset, RESULTS_ROOTS[dataset])
    fig.savefig(OUTPUT_DIR / f"figure_a9_11_correlation_{dataset}.png", dpi=200, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"figure_a9_11_correlation_{dataset}.pdf", bbox_inches="tight")
    print(f"Saved to {OUTPUT_DIR / f'figure_a9_11_correlation_{dataset}.png'}")
